# Stage 1 Pilot — `random` variant, seed-controlled LoRA runs

**Kaggle notebook — thin controller only. All logic lives in `scripts/run_pilot_experiment.py`
and `src/training/train_lora.py`.**

Runs the `random` arm of the Stage 1 pilot comparison (`data-stage-1.md §7`):
3 fresh LoRA fine-tunes of `Qwen/Qwen2.5-Coder-0.5B` on the `random` FIM
dataset variant, one per seed in `configs/training/pilot.yaml`, all logged to
the shared W&B project. Only the training data and seed vary — same
`configs/training/lora.yaml` hyperparameters as every other run.

Split into one notebook per variant (this one, and its `planned.ipynb`
sibling) so each Kaggle session only needs to fit 3 seeds' worth of training,
not all 6 — see `configs/training/pilot.yaml` and `data-stage-1.md §7-8` for
why 2-3 seeds per variant matters (a 10k-file pilot result is a signal, not
proof, and seed variance must be distinguishable from a real distribution
effect).

Data loads directly from Hugging Face (`load_dataset()`), no Kaggle dataset
attachment needed — `data-stage-1.md §7`.

## Required Kaggle setup before running
| Setting | Value |
|---|---|
| Accelerator | GPU T4 x2 |
| Internet | **ON** |
| Secret: `HF_TOKEN` | Hugging Face **write** token |
| Secret: `WANDB_API_KEY` | W&B API key from https://wandb.ai/authorize |


In [ ]:
# ── Cell 1: Clone repository at the requested Git state ──────────────────────
import os
import shutil
import subprocess
import sys

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/data"  # None -> main
COMMIT = None  # None -> latest commit on BRANCH

# Leave "" for normal behavior (each seed auto-resumes its own mid-training
# checkpoint/W&B run if this exact (variant, seed) was already in progress).
# Paste an exact run_id here ONLY to force-resume one specific crashed/
# interrupted seed instead -- e.g. copy it straight from that run's title in
# the W&B UI:
#   "qwen05b-lora-r16-e1-dsrandom-20260821-a42"
# This only affects the one seed whose identity matches (dataset_config +
# seed) -- every other seed in this run trains/resumes normally. Errors
# loudly if that run_id has no matching HF checkpoint, instead of silently
# training that seed from scratch.
RESUME_RUN_ID = ""

REPO_DIR = "/kaggle/working/qwen2.5-coder-0.5b-python-fim"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Cloning branch : {BRANCH or 'main'}")
print(f"Requested commit : {COMMIT or '(latest on branch)'}")

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [GITHUB_REPO, REPO_DIR]
subprocess.run(clone_cmd, check=True)

if COMMIT:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip()
current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

print(f"\n✓ Repository ready at {REPO_DIR}")
print(f"  Branch : {current_branch or '(detached HEAD)'}")
print(f"  Commit : {current_commit}")
if RESUME_RUN_ID:
    print(f"  Resume : {RESUME_RUN_ID}  (explicit — will force-resume this one seed)")


In [ ]:
# ── GPU check — stop early on a CPU-only session ─────────────────────────────
# This notebook needs a GPU (LoRA training via unsloth). If Kaggle assigned a
# CPU-only session (accelerator not set to GPU, or GPU quota exhausted), stop
# right here instead of burning the session on installs/training that will
# crash or run unusably slowly on CPU.
import subprocess

try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    print("\u2713 GPU detected \u2014 continuing.")
except (subprocess.CalledProcessError, FileNotFoundError):
    raise SystemExit(
        "\u2717 No GPU detected. This notebook requires a GPU \u2014 set the accelerator in "
        "Kaggle: Settings (right sidebar) \u2192 Accelerator \u2192 GPU T4 x2 (or similar), then "
        "Save & re-run. Stopping now to save your session quota."
    )


In [ ]:
# ── Cell 2: Install dependencies (GPU training + tracking) ───────────────────
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
        "peft>=0.20.0",
        "trl>=0.17.0",
        "transformers>=5.0.0",
        "datasets>=3.0.0",
        "pyarrow",
        "pyyaml>=6.0",
        "huggingface_hub>=0.30.0",
        "wandb>=0.19.0",
        "weave>=0.51.0",
    ],
    check=True,
)

print("✓ Dependencies installed")


In [ ]:
# ── Cell 3: Authenticate to Hugging Face + W&B ────────────────────────────────
# Both stored as Kaggle Secrets — NEVER hardcode tokens/keys.
# Add them: Kaggle account → Settings → Secrets → Add New Secret
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

print("✓ HF_TOKEN and WANDB_API_KEY loaded from Kaggle Secrets")


In [ ]:
# ── Cell 4: Run the "random" variant's pilot seeds ─────────────────────────
# Trains one fresh LoRA run per seed in configs/training/pilot.yaml, all from
# the base model (never continuing from a previous run) — see
# scripts/run_pilot_experiment.py. Each run is logged to W&B individually.
# Re-running this cell after an interruption (net/compute drop) is safe and
# resumes at the (variant, seed) level: run_pilot_experiment.run() checks HF
# commit history for each seed's deterministic run_id before training it, and
# skips straight to re-fetching that seed's already-pushed adapter (+ redoing
# just its SAFIM eval, since local results/pilot/... doesn't survive a session
# restart) instead of re-training from scratch. See module docstring in
# scripts/run_pilot_experiment.py for the full mechanism.
import subprocess
import sys

cmd = [sys.executable, "scripts/run_pilot_experiment.py", "--variant", "random"]
if RESUME_RUN_ID:
    cmd += ["--resume-run-id", RESUME_RUN_ID]

subprocess.run(cmd, check=True)


In [ ]:
# ── Cell 5: Where to look next ────────────────────────────────────────────────
print(
    "✓ 'random' variant pilot runs complete.\n"
    "  View them in the W&B project: "
    "https://wandb.ai/521er1007-national-institute-of-technology-rourkela/qwen-coder-python-fim\n"
    "  (filter by tag='random')\n\n"
    "  Once BOTH variant notebooks have finished (this one and planned.ipynb),\n"
    "  run: python scripts/analyze_pilot_results.py\n"
    "  to aggregate SAFIM pass@1 across seeds and check whether the "
    "random-vs-planned gap\n"
    "  is larger than seed-to-seed noise (data-stage-1.md §8)."
)
